**Please make sure you use "human_env" kernel**


Set base working directory

In [43]:
%cd /Users/peppinaloumoser/Downloads/notebook-2

/Users/peppinaloumoser/Downloads/notebook-2


---

### Manipulate VCF file with BCFtools and VCFtools

You can install BCFtools with conda

`conda install bcftools`

You can install VCFtools with conda

`conda install vcftools`


**Check what we get from last session**

View only header of Sample B GVCF

In [44]:
!bcftools view -h ./output/haplotypecaller/B.g.vcf

##fileformat=VCFv4.2
##FILTER=<ID=PASS,Description="All filters passed">
##ALT=<ID=NON_REF,Description="Represents any possible alternative allele not already represented at this location by REF and ALT">
##FILTER=<ID=LowQual,Description="Low quality">
##FORMAT=<ID=AD,Number=R,Type=Integer,Description="Allelic depths for the ref and alt alleles in the order listed">
##FORMAT=<ID=DP,Number=1,Type=Integer,Description="Approximate read depth (reads with MQ=255 or with bad mates are filtered)">
##FORMAT=<ID=GQ,Number=1,Type=Integer,Description="Genotype Quality">
##FORMAT=<ID=GT,Number=1,Type=String,Description="Genotype">
##FORMAT=<ID=MIN_DP,Number=1,Type=Integer,Description="Minimum DP observed within the GVCF block">
##FORMAT=<ID=PGT,Number=1,Type=String,Description="Physical phasing haplotype information, describing how the alternate alleles are phased in relation to one another; will always be heterozygous and is not intended to describe called alleles">
##FORMAT=<ID=PID,Number=1,Type

View only data

In [45]:
!bcftools view -H ./output/haplotypecaller/B.g.vcf | head -10

chr14	1	.	N	<NON_REF>	.	.	END=18601023	GT:DP:GQ:MIN_DP:PL	0/0:0:0:0:0,0,0
chr14	18601024	.	C	<NON_REF>	.	.	END=18601029	GT:DP:GQ:MIN_DP:PL	0/0:2:6:2:0,6,73
chr14	18601030	.	T	<NON_REF>	.	.	END=18601038	GT:DP:GQ:MIN_DP:PL	0/0:3:9:3:0,9,109
chr14	18601039	.	G	<NON_REF>	.	.	END=18601040	GT:DP:GQ:MIN_DP:PL	0/0:4:12:4:0,12,152
chr14	18601041	.	A	<NON_REF>	.	.	END=18601042	GT:DP:GQ:MIN_DP:PL	0/0:5:15:5:0,15,190
chr14	18601043	.	A	<NON_REF>	.	.	END=18601049	GT:DP:GQ:MIN_DP:PL	0/0:6:18:6:0,18,206
chr14	18601050	.	T	<NON_REF>	.	.	END=18601052	GT:DP:GQ:MIN_DP:PL	0/0:7:21:7:0,21,259
chr14	18601053	.	T	<NON_REF>	.	.	END=18601053	GT:DP:GQ:MIN_DP:PL	0/0:8:24:8:0,24,298
chr14	18601054	.	G	<NON_REF>	.	.	END=18601060	GT:DP:GQ:MIN_DP:PL	0/0:9:27:9:0,27,328
chr14	18601061	.	A	<NON_REF>	.	.	END=18601061	GT:DP:GQ:MIN_DP:PL	0/0:10:30:10:0,30,376
[main_vcfview] Error: cannot write to (null)


---
# GenomicsDBImport

Combine multiple GVCF files (Sample A + Sample B)


In [46]:
!gatk GenomicsDBImport -h

Using GATK jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar GenomicsDBImport -h
USAGE: GenomicsDBImport [arguments]

Import VCFs to GenomicsDB
Version:4.6.2.0


Required Arguments:

--genomicsdb-update-workspace-path <String>
                              Workspace when updating GenomicsDB. Can be a POSIX file system absolute or relative path
                              or a HDFS/GCS URL. Use this argument when adding new samples to an existing GenomicsDB
                              workspace or when using the output-interval-list-to-file option. Either this or
                              genomicsdb-workspace-path must be specified. Must point to an existing workspace. 
                

---
### Prepare sample name mapping file

It is a tab delimited file with 2 column

sampleID|path/to/sample.g.vcf

---

Use a combination of `ls` `xargs` `basename` `awk` to generate list file contain sample name.

`ls` will list all specifiy target

Both A.g.vcf and B.g.vcf should be listed here:

In [47]:
!ls ./output/haplotypecaller/*.g.vcf

./output/haplotypecaller/A.g.vcf ./output/haplotypecaller/B.g.vcf


---
A pipe character `|` has been use to conect two command

`xargs` will read standard input delimited by blanks or new line and executes the command.

`-n1` was use to limit 1 argument per command line

`basename`  take phrase path as input then returns the component
       following the final '/'.

In [48]:
!ls ./output/haplotypecaller/*.g.vcf | xargs -n1 basename

A.g.vcf
B.g.vcf


Use `awk` a text processing command to split file name.

Chain all commands with pipe `|` and save the result in text file.

This will generate input_name.txt containing both A and B:

In [49]:
!ls ./output/haplotypecaller/*.g.vcf | xargs -n1 basename \
    | awk '{split($0,a,".");print a[1]}' > ./output/haplotypecaller/input_name.txt

In [50]:
!cat ./output/haplotypecaller/input_name.txt

A
B


---
Use `realpath` command to create list file contain full path of target file

In [51]:
!realpath ./output/haplotypecaller/*.g.vcf > ./output/haplotypecaller/input_path.txt

In [52]:
!cat ./output/haplotypecaller/input_path.txt

/Users/peppinaloumoser/Downloads/notebook-2/output/haplotypecaller/A.g.vcf
/Users/peppinaloumoser/Downloads/notebook-2/output/haplotypecaller/B.g.vcf


Use `paste` command for join files horizontally.
Use `-d` option to set delimeter

This creates samples_map.txt with both Sample A and Sample B:

In [53]:
!paste ./output/haplotypecaller/input_name.txt \
    ./output/haplotypecaller/input_path.txt  > ./output/haplotypecaller/samples_map.txt

In [54]:
!cat ./output/haplotypecaller/samples_map.txt

A	/Users/peppinaloumoser/Downloads/notebook-2/output/haplotypecaller/A.g.vcf
B	/Users/peppinaloumoser/Downloads/notebook-2/output/haplotypecaller/B.g.vcf


---
**Create genomeDB with both Sample A and Sample B**

In [55]:
!gatk --java-options "-Xms8G -Xmx16G" GenomicsDBImport \
    --genomicsdb-workspace-path ./output/genomeDB \
    --batch-size 50 \
    -L chr14 \
    --sample-name-map ./output/haplotypecaller/samples_map.txt \
    --tmp-dir ./tmp \
    --reader-threads 2

Using GATK jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xms8G -Xmx16G -jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar GenomicsDBImport --genomicsdb-workspace-path ./output/genomeDB --batch-size 50 -L chr14 --sample-name-map ./output/haplotypecaller/samples_map.txt --tmp-dir ./tmp --reader-threads 2
17:18:33.624 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

17:18:33.633 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/7z/lvt4y45j06q3bp8mgvj0pq9w0000gn/T/libgkl_compression6421998010695666776.dylib: dlop

---
**(Additional) Update existing genomeDB**

You can add new sample to existing genomeDB by using the same command but with different option.

`--genomicsdb-update-workspace-path`

In [56]:
"""
!gatk --java-options "-Xms8G -Xmx16G" GenomicsDBImport \
    --genomicsdb-update-workspace-path ./output/genomeDB \
    --batch-size 50 \
    -L chr14 \
    --sample-name-map ./ouptut/haplotypecaller/new_samples_map.txt \
    --tmp-dir ./tmp \
    --reader-threads 2
"""

'\n!gatk --java-options "-Xms8G -Xmx16G" GenomicsDBImport     --genomicsdb-update-workspace-path ./output/genomeDB     --batch-size 50     -L chr14     --sample-name-map ./ouptut/haplotypecaller/new_samples_map.txt     --tmp-dir ./tmp     --reader-threads 2\n'

---
# GenotypeGVCFs

Perform joint genotyping on single or multiple sample pre-called with HaplotypeCaller

**Single sample — Sample B only**

In [57]:
!gatk GenotypeGVCFs -h

Using GATK jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -h
USAGE: GenotypeGVCFs [arguments]

Perform joint genotyping on a single-sample GVCF from HaplotypeCaller or a multi-sample GVCF from CombineGVCFs or
GenomicsDBImport
Version:4.6.2.0


Required Arguments:

--output,-O <GATKPath>        File to which variants should be written  Required. 

--reference,-R <GATKPath>     Reference sequence file  Required. 

--variant,-V <String>         A VCF file containing variants  Required. 


Optional Arguments:

--add-output-sam-program-record <Boolean>
                              If true, adds a PG tag to created SAM/BAM/CRAM files.  Default value: true. Possible
                

In [58]:
!mkdir -p ./output/genotypegvcf

In [59]:
!gatk --java-options "-Xmx8G" GenotypeGVCFs \
    -R ./reference/chr14.fa \
    -V ./output/haplotypecaller/B.g.vcf \
    -O ./output/genotypegvcf/B.vcf \
    --tmp-dir ./tmp

Using GATK jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -R ./reference/chr14.fa -V ./output/haplotypecaller/B.g.vcf -O ./output/genotypegvcf/B.vcf --tmp-dir ./tmp
17:18:48.742 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

17:18:48.754 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/7z/lvt4y45j06q3bp8mgvj0pq9w0000gn/T/libgkl_compression8586254241892722924.dylib: dlopen(/private/var/folders/7z/lvt4y45j06q3bp8mgvj0pq9w0000gn/T/libgkl_co

In [60]:
!bcftools view -H ./output/genotypegvcf/B.vcf | head -10

chr14	18601385	.	C	T	69.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-1.123;DP=61;ExcessHet=0;FS=23.489;MLEAC=1;MLEAF=0.5;MQ=35.53;MQRankSum=-0.143;QD=1.22;ReadPosRankSum=-0.706;SOR=4.167	GT:AD:DP:GQ:PL	0/1:50,7:57:77:77,0,1515
chr14	18601430	.	A	C	632.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.495;DP=83;ExcessHet=0;FS=73.721;MLEAC=1;MLEAF=0.5;MQ=33.47;MQRankSum=0.693;QD=8.55;ReadPosRankSum=-0.132;SOR=6.517	GT:AD:DP:GQ:PL	0/1:49,25:74:99:640,0,1470
chr14	18601553	.	T	A	3281.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-7.851;DP=284;ExcessHet=0;FS=30.232;MLEAC=1;MLEAF=0.5;MQ=38.36;MQRankSum=-4.714;QD=12.57;ReadPosRankSum=-2.149;SOR=2.105	GT:AD:DP:GQ:PL	0/1:139,122:261:99:3289,0,4044
chr14	18601712	.	G	T	1177.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.989;DP=94;ExcessHet=0;FS=3.04;MLEAC=1;MLEAF=0.5;MQ=44.67;MQRankSum=-8.252;QD=13.08;ReadPosRankSum=-1.088;SOR=2.144	GT:AD:DP:GQ:PL	0/1:48,42:90:99:1185,0,1576
chr14	18601815	.	T	G	43.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=2.64;DP=21;ExcessHet=0;FS=6.69;MLEAC=1;MLEAF=0.5;MQ=3

**Multiple sample — Combined Sample A + Sample B**

In [61]:
!gatk --java-options "-Xmx8G" GenotypeGVCFs \
    -R ./reference/chr14.fa \
    -V gendb://${PWD}/output/genomeDB \
    -O ./output/genotypegvcf/combine.vcf \
    --tmp-dir ./tmp

Using GATK jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar GenotypeGVCFs -R ./reference/chr14.fa -V gendb:///Users/peppinaloumoser/Downloads/notebook-2/output/genomeDB -O ./output/genotypegvcf/combine.vcf --tmp-dir ./tmp
17:19:01.862 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

17:19:01.871 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/7z/lvt4y45j06q3bp8mgvj0pq9w0000gn/T/libgkl_compression4185507096060862746.dylib: dlopen(/private/var/folders/7z/l


**View combined VCF file (should contain both sample A and B columns)**

In [62]:
!bcftools view -h ./output/genotypegvcf/combine.vcf | tail

##INFO=<ID=RAW_MQandDP,Number=2,Type=Integer,Description="Raw data (sum of squared MQ and total depth) for improved RMS Mapping Quality calculation. Incompatible with deprecated RAW_MQ formulation.">
##INFO=<ID=ReadPosRankSum,Number=1,Type=Float,Description="Z-score from Wilcoxon rank sum test of Alt vs. Ref read position bias">
##INFO=<ID=SOR,Number=1,Type=Float,Description="Symmetric Odds Ratio of 2x2 contingency table to detect strand bias">
##contig=<ID=chr14,length=107043718>
##source=GenomicsDBImport
##source=GenotypeGVCFs
##source=HaplotypeCaller
##bcftools_viewVersion=1.23.1+htslib-1.23.1
##bcftools_viewCommand=view -h ./output/genotypegvcf/combine.vcf; Date=Fri May  1 17:19:11 2026
#CHROM	POS	ID	REF	ALT	QUAL	FILTER	INFO	FORMAT	B


In [63]:
!bcftools view -H ./output/genotypegvcf/combine.vcf | head -10

chr14	18601385	.	C	T	69.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-1.123;DP=61;ExcessHet=0;FS=23.489;MLEAC=1;MLEAF=0.5;MQ=35.53;MQRankSum=-0.143;QD=1.22;ReadPosRankSum=-0.706;SOR=4.167	GT:AD:DP:GQ:PL	0/1:50,7:57:77:77,0,1515
chr14	18601430	.	A	C	632.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.495;DP=83;ExcessHet=0;FS=73.721;MLEAC=1;MLEAF=0.5;MQ=33.47;MQRankSum=0.693;QD=8.55;ReadPosRankSum=-0.132;SOR=6.517	GT:AD:DP:GQ:PL	0/1:49,25:74:99:640,0,1470
chr14	18601553	.	T	A	3281.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-7.851;DP=284;ExcessHet=0;FS=30.232;MLEAC=1;MLEAF=0.5;MQ=38.36;MQRankSum=-4.714;QD=12.57;ReadPosRankSum=-2.149;SOR=2.105	GT:AD:DP:GQ:PL	0/1:139,122:261:99:3289,0,4044
chr14	18601712	.	G	T	1177.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.989;DP=94;ExcessHet=0;FS=3.04;MLEAC=1;MLEAF=0.5;MQ=44.67;MQRankSum=-8.252;QD=13.08;ReadPosRankSum=-1.088;SOR=2.144	GT:AD:DP:GQ:PL	0/1:48,42:90:99:1185,0,1576
chr14	18601815	.	T	G	43.64	.	AC=1;AF=0.5;AN=2;BaseQRankSum=2.64;DP=21;ExcessHet=0;FS=6.69;MLEAC=1;MLEAF=0.5;MQ=3

---
# Variant Quality Score Recalibration (VQSR)

VQSR have 2 step.

    1. Create model with command `VariantRecalibrator`. Run separately for SNP and INDEL
    
    2. Apply model to data with command 'ApplyVQSR'

In [64]:
!gatk VariantRecalibrator -h

Using GATK jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar VariantRecalibrator -h
USAGE: VariantRecalibrator [arguments]

Build a recalibration model to score variant quality for filtering purposes
Version:4.6.2.0


Required Arguments:

--output,-O <GATKPath>        The output recal file used by ApplyVQSR  Required. 

--resource <FeatureInput>     A list of sites for which to apply a prior probability of being correct but which aren't
                              used by the algorithm (training and truth sets are required to run)  This argument must be
                              specified at least once. Required. 

--tranches-file <File>        The output tranches file used by ApplyVQSR 

In [65]:
!gatk ApplyVQSR -h

Using GATK jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar ApplyVQSR -h
USAGE: ApplyVQSR [arguments]

Apply a score cutoff to filter variants based on a recalibration table
Version:4.6.2.0


Required Arguments:

--output,-O <GATKPath>        The output filtered and recalibrated VCF file in which each variant is annotated with its
                              VQSLOD value  Required. 

--recal-file <FeatureInput>   The input recal file used by ApplyVQSR  Required. 

--variant,-V <GATKPath>       One or more VCF files containing variants  This argument must be specified at least once.
                              Required. 


Optional Arguments:

--add-output-sam-program-record <Boolean>
    

Create output directory

In [66]:
!mkdir -p ./output/vqsr

## Create model for SNP

Based on recommendation from GATK website

https://gatk.broadinstitute.org/hc/en-us/articles/4402736812443-Which-training-sets-arguments-should-I-use-for-running-VQSR

The standard set are store in

https://console.cloud.google.com/storage/browser/gcp-public-data--broad-references/hg38/v0?pageState=(%22StorageObjectListTable%22:(%22f%22:%22%255B%255D%22))

Running on combined A+B VCF:

In [67]:
!gatk --java-options "-Xmx8G" VariantRecalibrator \
    -R ./reference/chr14.fa \
    -V ./output/genotypegvcf/combine.vcf \
    --resource:hapmap,known=false,training=true,truth=true,prior=15.0 ./db/vqsr/resources_broad_hg38_v0_hapmap_3.3.hg38.vcf.gz  \
    --resource:omni,known=false,training=true,truth=false,prior=12.0 ./db/vqsr/resources_broad_hg38_v0_1000G_omni2.5.hg38.vcf.gz \
    --resource:1000G,known=false,training=true,truth=false,prior=10.0 ./db/vqsr/resources_broad_hg38_v0_1000G_phase1.snps.high_confidence.hg38_chr14roi.vcf.gz \
    --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
    -an QD -an MQ -an MQRankSum -an ReadPosRankSum -an FS -an SOR  \
    -mode SNP \
    --max-gaussians 4 \
    -O ./output/vqsr/vqsr_snp.recal \
    --tranches-file ./output/vqsr/vqsr_snp.tranches

Using GATK jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar VariantRecalibrator -R ./reference/chr14.fa -V ./output/genotypegvcf/combine.vcf --resource:hapmap,known=false,training=true,truth=true,prior=15.0 ./db/vqsr/resources_broad_hg38_v0_hapmap_3.3.hg38.vcf.gz --resource:omni,known=false,training=true,truth=false,prior=12.0 ./db/vqsr/resources_broad_hg38_v0_1000G_omni2.5.hg38.vcf.gz --resource:1000G,known=false,training=true,truth=false,prior=10.0 ./db/vqsr/resources_broad_hg38_v0_1000G_phase1.snps.high_confidence.hg38_chr14roi.vcf.gz --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz -an QD -an MQ -an MQRankSum -an Read

## Create model for INDEL

In [68]:
!gatk --java-options "-Xmx8G" VariantRecalibrator \
    -R ./reference/chr14.fa \
    -V ./output/genotypegvcf/combine.vcf \
    --resource:mills,known=false,training=true,truth=true,prior=12.0 ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz \
    --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz \
    -an QD -an MQRankSum -an ReadPosRankSum -an FS -an SOR -an DP \
    -mode INDEL \
    --max-gaussians 1 \
    -O ./output/vqsr/vqsr_indel.recal \
    --tranches-file ./output/vqsr/vqsr_indel.tranches

Using GATK jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar VariantRecalibrator -R ./reference/chr14.fa -V ./output/genotypegvcf/combine.vcf --resource:mills,known=false,training=true,truth=true,prior=12.0 ./db/resources_broad_hg38_v0_Mills_and_1000G_gold_standard.indels.hg38.vcf.gz --resource:dbsnp,known=true,training=false,truth=false,prior=2.0 ./db/Homo_sapiens_assembly38.dbsnp138_roi.vcf.gz -an QD -an MQRankSum -an ReadPosRankSum -an FS -an SOR -an DP -mode INDEL --max-gaussians 1 -O ./output/vqsr/vqsr_indel.recal --tranches-file ./output/vqsr/vqsr_indel.tranches
17:19:50.969 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/opt/anaconda3/envs/human_env/sh

## Apply VQSR SNP model

In [69]:
!gatk --java-options "-Xmx8G" ApplyVQSR \
  -V ./output/genotypegvcf/combine.vcf \
  --recal-file ./output/vqsr/vqsr_snp.recal \
  -mode SNP \
  --tranches-file ./output/vqsr/vqsr_snp.tranches \
  --truth-sensitivity-filter-level 99.5 \
  --create-output-variant-index true \
  -O ./output/vqsr/combine_snp_recalibrated.vcf.gz

Using GATK jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar ApplyVQSR -V ./output/genotypegvcf/combine.vcf --recal-file ./output/vqsr/vqsr_snp.recal -mode SNP --tranches-file ./output/vqsr/vqsr_snp.tranches --truth-sensitivity-filter-level 99.5 --create-output-variant-index true -O ./output/vqsr/combine_snp_recalibrated.vcf.gz
17:19:55.950 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

17:19:55.961 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/private/var/folders/7z/lvt

## Apply VQSR INDEL model

In [70]:
!gatk --java-options "-Xmx8G" ApplyVQSR \
  -V ./output/vqsr/combine_snp_recalibrated.vcf.gz \
  -mode INDEL \
  --recal-file ./output/vqsr/vqsr_indel.recal \
  --tranches-file ./output/vqsr/vqsr_indel.tranches \
  --truth-sensitivity-filter-level 99.0 \
  --create-output-variant-index true \
  -O ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz

Using GATK jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar
Running:
    java -Dsamjdk.use_async_io_read_samtools=false -Dsamjdk.use_async_io_write_samtools=true -Dsamjdk.use_async_io_write_tribble=false -Dsamjdk.compression_level=2 -Xmx8G -jar /opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar ApplyVQSR -V ./output/vqsr/combine_snp_recalibrated.vcf.gz -mode INDEL --recal-file ./output/vqsr/vqsr_indel.recal --tranches-file ./output/vqsr/vqsr_indel.tranches --truth-sensitivity-filter-level 99.0 --create-output-variant-index true -O ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz
17:20:00.518 INFO  NativeLibraryLoader - Loading libgkl_compression.dylib from jar:file:/opt/anaconda3/envs/human_env/share/gatk4-4.6.2.0-1/gatk-package-4.6.2.0-local.jar!/com/intel/gkl/native/libgkl_compression.dylib

17:20:00.525 WARN  NativeLibraryLoader - Unable to load libgkl_compression.dylib from native/libgkl_compression.dylib (/pr

### Gatk Hard filtering (Alternative)

Instead of using VQSR. GATK also provide a hard filtering setting.

https://gatk.broadinstitute.org/hc/en-us/articles/360035890471-Hard-filtering-germline-short-variants

In [71]:
"""
!gatk VariantFiltration \
     -V ./output/genotypegvcf/combine.vcf \
     -O ./output/genotypegvcf/combine_filtered_variants.vcf.gz \
     -filter "QD < 2.0" --filter-name "QD2" \
     -filter "MQ < 40.0" --filter-name "MQ40" \
     -filter "FS > 60.0" --filter-name "FS60" \
     -filter "SOR > 3.0" --filter-name "SOR3" \
     -filter "MQRankSum < -12.5" --filter-name "MQRankSum-12.5" \
     -filter "ReadPosRankSum < -8.0" --filter-name "ReadPosRankSum-8"
"""

'\n!gatk VariantFiltration      -V ./output/genotypegvcf/combine.vcf      -O ./output/genotypegvcf/combine_filtered_variants.vcf.gz      -filter "QD < 2.0" --filter-name "QD2"      -filter "MQ < 40.0" --filter-name "MQ40"      -filter "FS > 60.0" --filter-name "FS60"      -filter "SOR > 3.0" --filter-name "SOR3"      -filter "MQRankSum < -12.5" --filter-name "MQRankSum-12.5"      -filter "ReadPosRankSum < -8.0" --filter-name "ReadPosRankSum-8"\n'

## View your VCF file

In [72]:
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz | head -10

chr14	18601385	.	C	T	69.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-1.123;DP=61;ExcessHet=0;FS=23.489;MLEAC=1;MLEAF=0.5;MQ=35.53;MQRankSum=-0.143;QD=1.22;ReadPosRankSum=-0.706;SOR=4.167;VQSLOD=-78.74;culprit=FS	GT:AD:DP:GQ:PL	0/1:50,7:57:77:77,0,1515
chr14	18601430	.	A	C	632.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.495;DP=83;ExcessHet=0;FS=73.721;MLEAC=1;MLEAF=0.5;MQ=33.47;MQRankSum=0.693;QD=8.55;ReadPosRankSum=-0.132;SOR=6.517;VQSLOD=-752.7;culprit=FS	GT:AD:DP:GQ:PL	0/1:49,25:74:99:640,0,1470
chr14	18601553	.	T	A	3281.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-7.851;DP=284;ExcessHet=0;FS=30.232;MLEAC=1;MLEAF=0.5;MQ=38.36;MQRankSum=-4.714;QD=12.57;ReadPosRankSum=-2.149;SOR=2.105;VQSLOD=-132.4;culprit=FS	GT:AD:DP:GQ:PL	0/1:139,122:261:99:3289,0,4044
chr14	18601712	.	G	T	1177.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.989;DP=94;ExcessHet=0;FS=3.04;MLEAC=1;MLEAF=0.5;MQ=44.67;MQRankSum=-8.252;QD=13.08;ReadPosRankSum=-1.088;SOR=2.144

---

### Filter VCF with vcftools

In [73]:
!vcftools --gzvcf ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz \
    --recode \
    --recode-INFO-all \
    --remove-filtered-all \
    --out ./output/vqsr/combine_indel_snp_recalibrated


VCFtools - 0.1.17
(C) Adam Auton and Anthony Marcketta 2009

Parameters as interpreted:
	--gzvcf ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz
	--recode-INFO-all
	--out ./output/vqsr/combine_indel_snp_recalibrated
	--recode
	--remove-filtered-all

Using zlib version: 1.3.1
After filtering, kept 1 out of 1 Individuals
Outputting VCF file...
After filtering, kept 2610 out of a possible 2688 Sites
Run Time = 0.00 seconds


**BCFtools equivalent command**

In [74]:
"""
!bcftools view \
    -m 2 -M 2 \
    -f PASS \
    -O v \
    -o ./output/vqsr/combine_indel_snp_recalibrated.vcf \
    ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz
"""

'\n!bcftools view     -m 2 -M 2     -f PASS     -O v     -o ./output/vqsr/combine_indel_snp_recalibrated.vcf     ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz\n'

### Check VCF file before and after filtering

In [75]:
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.vcf.gz | head -10

chr14	18601385	.	C	T	69.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-1.123;DP=61;ExcessHet=0;FS=23.489;MLEAC=1;MLEAF=0.5;MQ=35.53;MQRankSum=-0.143;QD=1.22;ReadPosRankSum=-0.706;SOR=4.167;VQSLOD=-78.74;culprit=FS	GT:AD:DP:GQ:PL	0/1:50,7:57:77:77,0,1515
chr14	18601430	.	A	C	632.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-5.495;DP=83;ExcessHet=0;FS=73.721;MLEAC=1;MLEAF=0.5;MQ=33.47;MQRankSum=0.693;QD=8.55;ReadPosRankSum=-0.132;SOR=6.517;VQSLOD=-752.7;culprit=FS	GT:AD:DP:GQ:PL	0/1:49,25:74:99:640,0,1470
chr14	18601553	.	T	A	3281.64	VQSRTrancheSNP99.90to100.00	AC=1;AF=0.5;AN=2;BaseQRankSum=-7.851;DP=284;ExcessHet=0;FS=30.232;MLEAC=1;MLEAF=0.5;MQ=38.36;MQRankSum=-4.714;QD=12.57;ReadPosRankSum=-2.149;SOR=2.105;VQSLOD=-132.4;culprit=FS	GT:AD:DP:GQ:PL	0/1:139,122:261:99:3289,0,4044
chr14	18601712	.	G	T	1177.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.989;DP=94;ExcessHet=0;FS=3.04;MLEAC=1;MLEAF=0.5;MQ=44.67;MQRankSum=-8.252;QD=13.08;ReadPosRankSum=-1.088;SOR=2.144

In [76]:
!bcftools view -H ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf | head -10

chr14	18601712	.	G	T	1177.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=-3.989;DP=94;ExcessHet=0;FS=3.04;MLEAC=1;MLEAF=0.5;MQ=44.67;MQRankSum=-8.252;QD=13.08;ReadPosRankSum=-1.088;SOR=2.144;VQSLOD=12.56;culprit=MQRankSum	GT:AD:DP:GQ:PL	0/1:48,42:90:99:1185,0,1576
chr14	18601815	.	T	G	43.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=2.64;DP=21;ExcessHet=0;FS=6.69;MLEAC=1;MLEAF=0.5;MQ=32.21;MQRankSum=1.76;QD=2.08;ReadPosRankSum=0;SOR=2.814;VQSLOD=33.44;culprit=QD	GT:AD:DP:GQ:PL	0/1:18,3:21:51:51,0,699
chr14	18967563	.	G	A	51.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=0.179;DP=80;ExcessHet=0;FS=0;MLEAC=1;MLEAF=0.5;MQ=25.39;MQRankSum=-1.207;QD=0.65;ReadPosRankSum=-0.037;SOR=0.024;VQSLOD=93.02;culprit=QD	GT:AD:DP:GQ:PL	0/1:69,10:79:59:59,0,1846
chr14	18967635	.	T	C	1446.06	PASS	AC=2;AF=1;AN=2;DP=67;ExcessHet=0;FS=0;MLEAC=2;MLEAF=1;MQ=32.38;QD=30.77;SOR=7.743;VQSLOD=27.91;culprit=MQ	GT:AD:DP:GQ:PL	1/1:0,47:47:99:1460,141,0
chr14	18999587	.	T	C	485.64	PASS	AC=1;AF=0.5;AN=2;BaseQRankSum=-6.375;DP=89;ExcessHet=0;FS

---
**Compress and index VCF file**

In [77]:
!bgzip -c ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf > ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz

In [78]:
!tabix -p vcf ./output/vqsr/combine_indel_snp_recalibrated.recode.vcf.gz

---
**Pipeline complete for Sample B!**

The final combined VCF `combine_indel_snp_recalibrated.recode.vcf.gz` contains variants from both Sample A and Sample B, filtered and recalibrated.
